# Unit 07 · Simulated vs real

Everything so far was trained and tested on **DRRs**, X-rays we simulated from CTs. Here the network meets **real
fluoroscopy**: DeepFluoro, six cadavers with a CT each and hundreds of real C-arm frames whose camera poses are
known. How much worse does it get, and why? Then: can we check a prediction without a ground truth (re-projection,
uncertainty), and where does the network fail on the simulated test set (error analysis)?

| Part | After video 13 … | What you build | Suggested driver |
|---|---|---|---|
| A | first half · real frames, the domain gap | the log transform, the gap | Adir |
| B | second half · re-projection, MC dropout | re-projection IoU, MC-dropout prediction, uncertainty vs error | Noa / Adir / Noa |
| C | end · error analysis | the clinical-angle error, the error summary | Noa |

**Needs:** units 01–05 moved into `src/xray3d` (`physics.hu_to_mu`, `geometry.project_pinhole`, `shape.*`,
`metrics.*`, `baselines.pixel_index`, `nets`, `losses`, `training`), the processed dataset v1 and
`splits/split_v1.json`, and the trained models of **unit 06** (downloaded from Kaggle, see below). DeepFluoro on disk
in `data/raw/deepfluoro`, plus `pip install pydicom` and DiffDRR (`pip install -e ".[drr]"`).

> **Licence: DeepFluoro is CC BY-NC-ND 4.0.** Nothing derived from it may be committed: no figure, no array, no
> per-cadaver number in a file. This notebook shows DeepFluoro images on screen only; its cached arrays go to
> `deepfluoro.cache_dir()` under the (git-ignored) data folder; the only file written into the repo is
> `results/sim_vs_real_v1.csv`, with aggregate numbers. **Before you commit: Kernel → Restart & Clear All Outputs,
> then save.** (TotalSegmentator images, CC BY, may be shown and kept.)

In [ ]:
# --- setup (written by Claude, no need to change it): the same cell works locally, on Colab and on Kaggle ---
import os, sys, subprocess
from pathlib import Path

UNIT = "07-sim-vs-real"
REPO_URL = "https://github.com/AdirBuskila/pelvis-xray-3d"
IN_CLOUD = "google.colab" in sys.modules or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_CLOUD:
    root = Path("/content" if "google.colab" in sys.modules else "/kaggle/working") / "pelvis-xray-3d"
    if not root.exists():  # Kaggle: turn Internet on in the notebook settings first
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
else:  # local: the repo is the first folder above this notebook that has a pyproject.toml
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root / "units" / UNIT)
for p in (root / "src", Path.cwd()):  # import xray3d (even before `pip install -e .`) and this unit's checks
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import xray3d
from xray3d import data, env, plotting
import test_unit as T

env.summary()

In [ ]:
import torch
from scipy import ndimage
from xray3d import dataset, deepfluoro, metrics, prep, torchdata, trainer
from xray3d.geometry import project_pinhole

# --- the trained models (unit 06's run names). The real evaluation uses unit 06's Kaggle runs: download that
# notebook's output (the runs/ folder) and unzip it into the folder printed below. Several seeds are averaged.
RUNS = {1: ["exp-ap-s0", "exp-ap-s1", "exp-ap-s2"],                  # 1 view (AP)
        2: ["exp-ap_lat-s0", "exp-ap_lat-s1", "exp-ap_lat-s2"]}      # 2 views (AP + LAT)
SMOKE = {1: "u07-smoke-1view", 2: "u07-smoke-2view"}                 # fallback: tiny CPU runs (numbers mean NOTHING)
MC_PASSES = 20                                                       # MC-dropout passes per prediction

print("runs folder:", trainer.runs_root())
if env.where() != "local" and not deepfluoro.specimens():
    deepfluoro.fetch()                     # Colab/Kaggle: ~400 MB from Hugging Face (only what this unit needs)
SPECIMENS = deepfluoro.specimens()
print("DeepFluoro specimens on disk:", SPECIMENS, " frames per specimen:", [len(deepfluoro.frames(s)) for s in SPECIMENS])

---
## Part A · The gap (after video 13, first half)

### A real frame
A C-arm frame is **not** a line integral. The detector counts photons: bright = many photons got through, so bone is
*dark*. The DICOM holds those counts (after the detector's own processing). Our network was trained on
`p = ∫ μ dl`, where bone is *bright*. Beer–Lambert (unit 01) links the two: `I = I0 · exp(−p)`.

Two numbers per frame are needed to undo it, and a real detector gives you neither for free:
- **dark**: what a pixel reads with **no** photons (electronic offset). The collimator blocks the beam completely at
  the frame's edge, so those pixels read the dark level.
- **flat** (`I0`): what a pixel reads with **nothing** in the beam. A pelvis frame has almost no open field, so it's a
  guess (`deepfluoro.dark_flat` takes the 99.9th percentile).

In [ ]:
sp0 = SPECIMENS[0]
table = deepfluoro.frame_table(sp0)
ang = np.array([r["angle_deg"] for r in table]); cov = np.array([r["coverage"] for r in table])
print(f"{sp0}: {len(table)} frames, angle to the A-P axis {ang.min():.1f}-{ang.max():.1f} deg, "
      f"pelvis coverage {cov.min():.0%}-{cov.max():.0%}")
fr0, a0 = deepfluoro.choose_frame(sp0)
print(f"chosen: frame {fr0} ({a0:.1f} deg): within 12 deg of A-P, the one that sees the most pelvis")
raw0 = deepfluoro.load_raw(sp0, fr0)
dark0, flat0 = deepfluoro.dark_flat(raw0)
print(f"raw frame {raw0.shape}, values {raw0.min():.0f}..{raw0.max():.0f};  dark {dark0:.0f}, flat {flat0:.0f}")
print("fraction of the frame at the dark level:", f"{(raw0 <= dark0).mean():.1%}", " (the collimated border)")
plotting.show_row([raw0, deepfluoro.crop(raw0)], ["the whole DICOM frame (bone dark)", "cropped to the 1436 px the pose describes"], size=4)

### TODO A1 · `log_transform`  ·  suggested driver: **Adir**

Turn detector values into the line integral the network knows: subtract the dark level from the pixel **and** from
the flat level, divide, take `−ln`. Pixels at or below dark (no photons) must not give `inf`; pixels brighter than
flat must not give a negative `p`.

> **Nudge:** solve `raw − dark = (flat − dark) · exp(−p)` for p. Clip the fraction to [eps, 1] before the log. The check also asks why an error in *flat* doesn't matter for our network but an error in *dark* does.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def log_transform(raw, dark, flat, eps=1e-6):
    """Detector values -> line integral p (Beer-Lambert, inverted).

    raw  : array of detector values (any shape); brighter = more photons
    dark : the value of a pixel that got no photons;  flat : the value with nothing in the beam (I0)
    eps  : the smallest fraction (raw - dark) / (flat - dark) allowed, so no pixel gives inf
    returns : p, same shape, float, >= 0 (bone large)
    """
    raise NotImplementedError("TODO A1")

In [ ]:
T.check_log_transform(log_transform)

### Building the six specimens

For each cadaver, `deepfluoro.load_specimen` (Claude's code, calling **yours**: `hu_to_mu`, `bbox_center_mm`,
`occupancy`, `cube_grid`, `resample`, and now `log_transform`) picks a frame, makes the ground-truth cube from the
DeepFluoro labels (L/R hemipelvis + upper/lower sacrum = TotalSegmentator's hips + sacrum), renders a DRR at the frame's
**real pose** with DiffDRR, the simulated AP + LAT pair exactly like the training data, and brings everything into
our network's geometry. ~40 s per specimen the first time, then it's cached (outside git).

In [ ]:
specs = {}
for sp in SPECIMENS:
    specs[sp] = deepfluoro.load_specimen(sp, log_transform)
s0 = specs[sp0]
p_real = log_transform(s0["raw_small"], s0["info"]["dark"], s0["info"]["flat"])
plotting.show_row([s0["raw_small"], p_real, s0["drr_small"]],
                  ["real: detector values", "real: YOUR log transform", "DRR at the same pose (whole CT)"], size=4)

**Is the real detector linear in `p`?** If the real frame were a perfect X-ray of the same CT, its `p` would equal
the DRR's `p` pixel by pixel, up to a scale and an offset (which `standardise` removes). Each dot below is one detector
pixel of the frame above.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for sp in SPECIMENS:
    s = specs[sp]
    pr = log_transform(s["raw_small"], s["info"]["dark"], s["info"]["flat"]).ravel()
    pd = s["drr_small"].ravel()
    r_log, r_raw = np.corrcoef(pr, pd)[0, 1], np.corrcoef(-s["raw_small"].ravel(), pd)[0, 1]
    print(f"{sp}: corr(real p, DRR p) = {r_log:.3f}   corr(-raw, DRR p) = {r_raw:.3f}")
    if sp == sp0:
        ax[0].hexbin(pd, pr, gridsize=60, bins="log", cmap="magma"); ax[0].set(xlabel="DRR p", ylabel="real p (log transform)", title=sp)
        ax[1].hexbin(pd, s["raw_small"].ravel(), gridsize=60, bins="log", cmap="magma"); ax[1].set(xlabel="DRR p", ylabel="real raw value", title="without the log")
plt.tight_layout(); plt.show()

**Look before you trust the physics.** On DeepFluoro the *raw* values (sign flipped) follow the DRR's `p` as
straight a line as your log transform does, or straighter: the log bends it. The C-arm's image chain (image
intensifier, camera, video processing) had most likely already applied a log-like curve before the frame was saved,
so these DICOMs are not photon counts. Your `log_transform` is the right physics for a detector that stores counts
(and xvr, the code that registered these poses, applies it too); here it adds a mild extra curve. That's a gap too:
**what a "real X-ray file" contains depends on the machine.** The rest of the notebook uses your log transform. To
try the other choice: `deepfluoro.load_specimen(sp, lambda raw, dark, flat: -raw, rebuild=True)`.

### Into the network's geometry

The network only knows one camera: source **750 mm in front** of the pelvis centre, detector 1,020 mm from the source,
128 px of 4 mm, the cube's axes R/A/S. A real frame differs in four ways (details in the docstring of
`xray3d/deepfluoro.py`):

| | real frame | what we do | exact? |
|---|---|---|---|
| 1 direction | C-arm a few degrees off the CT's A-P axis | turn the **cube** (and the ground truth with it) so its A axis lies on the line source → pelvis centre | yes: a rotation about the source is a homography |
| 2 side | the source is **behind** the cadaver (PA) | mirror the image through the pelvis's coronal plane | no: front and back swap which one is magnified more (a few %) |
| 3 distance | source 650–830 mm from the pelvis centre | zoom by d / 750 | at the centre's depth only (~1 % off at ±100 mm) |
| 4 field of view | a 278 mm detector: ~200 mm at the pelvis | mark the rest of our 128 px **invalid**, fill it with the mean | can't be fixed |

So instead of one "sim vs real" number we can climb a ladder, one difference at a time (`deepfluoro.INPUTS`):

1. `sim_full`: a DRR in exact AP geometry, full field: what the network was trained on;
2. `sim_cut`: the same image, cut to the real field of view (→ the cost of **4**);
3. `drr_pose`: a DRR at the **real** pose, warped exactly like the real frame (→ the cost of **1–3**'s approximations);
4. `real`: the real frame, warped (→ the **appearance** gap: scatter, noise, the detector, the real tissue and table
   that the cropped CT doesn't have).

In [ ]:
for sp in SPECIMENS:
    i = specs[sp]["info"]; v = specs[sp]["valid"]
    a, b, c = (deepfluoro.ap_image(specs[sp], n)[v] for n in ("sim_cut", "drr_pose", "real"))
    print(f"{sp}: frame {i['frame']}, {i['cube_angle_deg']:4.1f} deg off A-P, source {i['source_to_center_mm']:.0f} mm behind "
          f"(zoom {i['zoom']:.3f}), field = {i['valid_fraction']:.0%} of our image;  "
          f"corr sim/drr_pose {np.corrcoef(a, b)[0, 1]:.2f}, drr_pose/real {np.corrcoef(b, c)[0, 1]:.2f}")
names = list(deepfluoro.INPUTS)
plotting.show_row([torchdata.standardise(deepfluoro.ap_image(s0, n)) for n in names] + [torchdata.standardise(s0["sim_lat"])],
                  [f"{k + 1}. {n}" for k, n in enumerate(names)] + ["simulated LAT (both 2-view inputs)"], size=3.3, vmin=-2.5, vmax=2.5)

### The models

The real evaluation uses unit 06's runs (3 seeds per arm, 4,000 steps on a GPU). If they aren't in the runs folder,
the next cell trains two **tiny** models on the CPU so that every cell below runs: their numbers mean **nothing**
(60 steps), and the results file says so.

In [ ]:
def available(names):
    return [n for n in names if (trainer.run_dir(n) / "best.pt").exists()]

USED, IS_SMOKE = {}, False
for v in (1, 2):
    USED[v] = available(RUNS[v])
    if not USED[v]:
        IS_SMOKE = True
        if not available([SMOKE[v]]):
            trainer.train(SMOKE[v], n_views=v, iters=60, batch_size=4, lr=3e-3, eval_every=30, ckpt_every=30, verbose=False)
        USED[v] = [SMOKE[v]]
MODELS = {v: [trainer.load_model(n) for n in USED[v]] for v in (1, 2)}
print("models used:", USED)
if IS_SMOKE:
    print("\n*** SMOKE MODELS (60 CPU steps): every number below is MEANINGLESS. Get unit 06's runs for the real evaluation. ***")

def predict(model, x):
    """Occupancy (64^3 bool) of one input (1, V, 128, 128): sigmoid > 0.5 is logit > 0."""
    model.eval()
    with torch.no_grad():
        return (model(x)[0, 0] > 0).numpy()

In [ ]:
METRICS = ("dice", "hd95", "nsd_5mm")
LADDER = {1: list(deepfluoro.INPUTS), 2: list(deepfluoro.INPUTS)}     # the 2-view model always gets the simulated LAT
scores = {}        # (n_views, input name) -> {specimen: {metric: mean over the seeds}}
preds = {}         # (n_views, input name, specimen) -> occupancy of the first model (for the pictures)
for v in (1, 2):
    for name in LADDER[v]:
        per = {}
        for sp in SPECIMENS:
            x = deepfluoro.network_input(specs[sp], name, v)
            ms = []
            for k, m in enumerate(MODELS[v]):
                occ = predict(m, x)
                if k == 0:
                    preds[v, name, sp] = occ
                ms.append(metrics.evaluate(specs[sp]["occ"], occ))
            per[sp] = {mt: float(np.mean([e[mt] for e in ms])) for mt in METRICS}
        scores[v, name] = per
        print(f"{v} view(s), {name:<9}: " + "  ".join(f"{mt} {np.mean([per[s][mt] for s in SPECIMENS]):6.3f}" for mt in METRICS))

### TODO A2 · `gap`  ·  suggested driver: **Adir**

The **gap** of one comparison: for every specimen measured both ways, how much worse the metric got from the simulated
input to the real one, then the mean, the standard deviation and the number of specimens that got worse. Positive must
mean *worse*, whatever the metric: a Dice drop is `sim − real`, an HD95 drop is `real − sim`.

> **Nudge:** pair by specimen (a specimen in only one dict, or with a nan, can't be paired). Six cadavers: is the sd across specimens big next to the mean? Use the n − 1 standard deviation (`ddof=1`); with one specimen it's nan.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def gap(metric_sim, metric_real, higher_is_better=True):
    """The drop of a metric from simulated to real input, per specimen and summarised.

    metric_sim, metric_real : {specimen: value}
    higher_is_better        : True for Dice / NSD, False for distances (HD95 in mm)
    returns {"per_specimen": {specimen: drop}, "n": int, "mean": float, "sd": float (n - 1; nan if n < 2),
             "worse": number of specimens with drop > 0}      drop > 0 = worse on the real side
    """
    raise NotImplementedError("TODO A2")

In [ ]:
T.check_gap(gap)

In [ ]:
STEPS = [("sim_full", "sim_cut", "field of view"), ("sim_cut", "drr_pose", "geometry approximations"),
         ("drr_pose", "real", "appearance (the domain gap)"), ("sim_full", "real", "total")]
HIGHER = {"dice": True, "hd95": False, "nsd_5mm": True}
GAPS = []
for v in (1, 2):
    print(f"--- {v} view(s){' (+ simulated LAT)' if v == 2 else ''}: drop from A to B, mean ± sd over {len(SPECIMENS)} specimens (worse in k)")
    for a, b, what in STEPS:
        line = f"  {a:>8} -> {b:<8} {what:<28}"
        for mt in METRICS:
            g = gap({s: d[mt] for s, d in scores[v, a].items()}, {s: d[mt] for s, d in scores[v, b].items()}, HIGHER[mt])
            GAPS.append({"n_views": v, "from": a, "to": b, "step": what, "metric": mt, **{k: g[k] for k in ("n", "mean", "sd", "worse")}})
            line += f"  {mt} {g['mean']:+7.3f} ± {g['sd']:.3f} ({g['worse']})"
        print(line)

In [ ]:
sp = sp0
gt = specs[sp]["occ"]
row = []
for v, name in [(1, "drr_pose"), (1, "real"), (2, "drr_pose"), (2, "real")]:
    row.append(prep.overlap_image(prep.coronal(gt), prep.coronal(preds[v, name, sp], j=int(np.argmax(gt.sum(axis=(0, 2)))))))
plotting.show_row(row, ["1 view, DRR at the pose", "1 view, real", "2 views, DRR at the pose + sim LAT", "2 views, real + sim LAT"], size=3.5)
print("white: right, red: predicted but not there, blue: missed (one coronal layer)")

**Explain-back (part A):** which rung of the ladder costs the most, for 1 view and for 2? Does the simulated LAT
protect the 2-view model from the real AP's problems, and why might it? The correlation between the real `p` and the
DRR `p` is high but not 1: name three physical reasons. Six cadavers and one frame each: how sure can you be of the
*sign* of a gap whose sd is as big as its mean? What would you change to measure the appearance gap without the
field-of-view problem?

**Part A done.** Move `log_transform` and `gap` into `src/xray3d/realxray.py`.

---
## Part B · Checking a prediction without a 3D ground truth (after video 13, second half)

On a real patient there is no CT, so no 3D ground truth. Two things can still be checked: does the predicted shape
**cast the shadow we saw** (re-projection), and how **sure** was the network (MC dropout)?

### TODO B1 · `reprojection_iou`  ·  suggested driver: **Noa**

Project the predicted occupancy into a view and compare its shadow with a silhouette from that view: the IoU of the two
masks, counted only inside `valid` (the real detector's field). Use **your** unit 03 `baselines.pixel_index` for the
projection. A 5 mm voxel lands ~1.7 px apart from its neighbour on our detector: one point per voxel leaves holes.

> **Nudge:** spread `sub`³ points evenly inside each occupied voxel (the voxel centres are `(i + 0.5)·cell − cube/2`), project them all, mark the pixels they hit; IoU = |A ∩ B| / |A ∪ B| inside `valid`. Nothing vs nothing is 1.0.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def reprojection_iou(occ, silhouette, camera, valid=None, cube_mm=320.0, sub=3):
    """IoU between the shadow of an occupancy cube in one view and a 2D silhouette.

    occ        : (n, n, n) occupancy in the cube frame (RAS axes, centred on the origin, side cube_mm)
    silhouette : (H, W) bool, e.g. ap_bone > 0, or the real frame's 2D labels
    camera     : (K, R, t) as dataset.camera(view) returns
    valid      : (H, W) bool, the pixels that count (default: all)
    sub        : points per voxel along each axis
    returns    : float in [0, 1]
    """
    raise NotImplementedError("TODO B1")

In [ ]:
T.check_reprojection_iou(reprojection_iou)

In [ ]:
cams = {v: dataset.camera(v) for v in ("ap", "lat")}
test_ids = dataset.subjects("test")
ts = {sid: dataset.load_subject(sid) for sid in test_ids}
ious = [[reprojection_iou(ts[s]["occ"], ts[s][f"{v}_bone"] > 1e-3, cams[v]) for v in ("ap", "lat")] for s in test_ids]
print(f"TotalSegmentator test ({len(test_ids)}): GT re-projected vs the pelvis-only DRR: AP {np.mean(ious, 0)[0]:.3f}, LAT {np.mean(ious, 0)[1]:.3f}"
      "  (the unit 02 acceptance test: >= 0.9)")
print("DeepFluoro: in each specimen's turned cube, our AP camera is dataset.camera('ap') again.")
REPROJ = []
for sp in SPECIMENS:
    s = specs[sp]
    g_iou = reprojection_iou(s["occ"], s["labels2d"], cams["ap"], valid=s["valid"])
    p_iou = reprojection_iou(preds[2, "real", sp], s["labels2d"], cams["ap"], valid=s["valid"])
    REPROJ.append({"specimen": sp, "gt": g_iou, "pred": p_iou, "dice": scores[2, "real"][sp]["dice"]})
    print(f"  {sp}: GT cube vs the frame's own 2D labels {g_iou:.3f}   2-view prediction (real AP) vs 2D labels {p_iou:.3f}"
          f"   (its 3D Dice {REPROJ[-1]['dice']:.3f})")

The first DeepFluoro number is a check of **our geometry** (and of the dataset's registered pose): the true 3D labels,
pushed through all the approximations, against labels drawn on the frame itself. It is the ceiling for the second
number: a perfect prediction can't re-project better than the ground truth does.

### TODO B2 · `mc_dropout_predict`  ·  suggested driver: **Adir**

**MC dropout:** keep the dropout layers *on* at test time and run the same input `n` times. Each pass is a slightly
different network; where they disagree, the model is unsure. Return the mean probability and the variance per voxel.
Only dropout may be switched on: normalisation layers must stay in eval mode.

> **Nudge:** `model.eval()` first, then walk `model.modules()` and call `.train()` only on the dropout layers (`torch.nn.Dropout`, `Dropout2d`, `Dropout3d`). Stack the n sigmoid outputs under `torch.no_grad()`; at the end put the whole model back in eval mode.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def mc_dropout_predict(model, x, n=20):
    """Mean and variance of sigmoid(model(x)) over n passes with dropout ON (everything else in eval mode).

    x       : the input batch, e.g. (1, V, 128, 128)
    returns : (mean, var), both shaped like the model's output, e.g. (1, 1, 64, 64, 64); no gradients
    The model is left in eval mode.
    """
    raise NotImplementedError("TODO B2")

In [ ]:
T.check_mc_dropout_predict(mc_dropout_predict)

### TODO B3 · `uncertainty_vs_error`  ·  suggested driver: **Noa**

Is the uncertainty **useful**? Per voxel: the error is 1 where the thresholded mean prediction (> 0.5) disagrees with
the ground truth, else 0. Return the correlation between variance and error, and the mean variance on wrong and on
right voxels. A `region` restricts it to the voxels near the pelvis: 90 % of the cube is trivially empty and would
make any uncertainty look good.

> **Nudge:** Pearson's r between a number and a 0/1 variable (`np.corrcoef`) is fine. If either is constant (e.g. no wrong voxel at all), r is undefined: nan.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def uncertainty_vs_error(var, mean_prob, gt, region=None):
    """Does the per-voxel variance point at the errors?

    var, mean_prob : arrays from mc_dropout_predict (numpy, any shape);  gt : 0/1 array of the same shape
    region         : bool array, the voxels to use (default: all)
    returns {"r": Pearson corr(var, error), "var_wrong": mean var where wrong, "var_right": mean var where right,
             "n": voxels used}      nan where undefined
    """
    raise NotImplementedError("TODO B3")

In [ ]:
T.check_uncertainty_vs_error(uncertainty_vs_error)

In [ ]:
model2 = MODELS[2][0]
def near_pelvis(gt, occ):
    return ndimage.binary_dilation(gt.astype(bool) | occ.astype(bool), iterations=2)

UNC = {"ts": [], "df": []}
for sid in test_ids:
    x = torch.from_numpy(torchdata.standardise(np.stack([ts[sid]["ap"], ts[sid]["lat"]])))[None]
    mean, var = (a[0, 0].numpy() for a in mc_dropout_predict(model2, x, MC_PASSES))
    u = uncertainty_vs_error(var, mean, ts[sid]["occ"], region=near_pelvis(ts[sid]["occ"], mean > 0.5))
    UNC["ts"].append({**u, "mean_var": float(var[near_pelvis(ts[sid]["occ"], mean > 0.5)].mean())})
print(f"TotalSegmentator test, 2 views: r(var, error) near the pelvis = {np.nanmean([u['r'] for u in UNC['ts']]):.3f} "
      f"(mean var: wrong {np.nanmean([u['var_wrong'] for u in UNC['ts']]):.4f}, right {np.nanmean([u['var_right'] for u in UNC['ts']]):.4f})")
for sp in SPECIMENS:
    row = {}
    for name in ("drr_pose", "real"):
        mean, var = (a[0, 0].numpy() for a in mc_dropout_predict(model2, deepfluoro.network_input(specs[sp], name, 2), MC_PASSES))
        reg = near_pelvis(specs[sp]["occ"], mean > 0.5)
        row[name] = {**uncertainty_vs_error(var, mean, specs[sp]["occ"], region=reg), "mean_var": float(var[reg].mean())}
    UNC["df"].append(row)
    print(f"  {sp}: mean var near the pelvis: DRR at pose {row['drr_pose']['mean_var']:.4f}, real {row['real']['mean_var']:.4f};"
          f"  r(var, error): {row['drr_pose']['r']:.3f} / {row['real']['r']:.3f}")
sid = test_ids[0]
x = torch.from_numpy(torchdata.standardise(np.stack([ts[sid]["ap"], ts[sid]["lat"]])))[None]
mean, var = (a[0, 0].numpy() for a in mc_dropout_predict(model2, x, MC_PASSES))
err = (mean > 0.5) != ts[sid]["occ"].astype(bool)
j = int(np.argmax(ts[sid]["occ"].sum(axis=(0, 2))))
plotting.show_row([prep.coronal(ts[sid]["occ"], j), prep.coronal(mean, j), prep.coronal(var, j), prep.coronal(err, j)],
                  [f"{sid}: truth", "MC mean probability", "MC variance", "error (mean > 0.5 vs truth)"], size=3.4, cmap="magma")

**Explain-back (part B):** why must the normalisation layers stay in eval mode during MC dropout? Our network uses
InstanceNorm: what does it do in train mode vs eval mode (does it keep running statistics?), so would `model.train()`
change its output at all? What would it do to a batch of one if it were BatchNorm? Is a high variance where the network is *wrong*, or only at the pelvis's *border*, where any small
change flips a voxel? Does the variance go up on the real frames, i.e. could it warn you that an input is unlike the
training data? Why is the re-projection IoU of the **ground truth** below 1, and what does that tell you about trusting
a re-projection check?

**Part B done.** Move `reprojection_iou`, `mc_dropout_predict` and `uncertainty_vs_error` into `src/xray3d/realxray.py`.

---
## Part C · Error analysis (after video 13, end)

Back to the simulated **test set** (9 TotalSegmentator subjects locally, ~60 with the full dataset), where every
answer is known. A mean Dice hides *which* patients fail and *how*. A surgeon cares about the hip socket, not the
voxel count: unit 02's clinical **CE angle** measures it.

### TODO C1 · `ce_error`  ·  suggested driver: **Noa**

The clinical error of one prediction: the absolute difference between the CE angle measured on the prediction and on
the ground truth, for each hip, at the **true** femoral head (both measurements use the same head; only the bone
changes). A missing head, or an angle that can't be measured, gives nan, never 0.

> **Nudge:** `heads` rows are (left, right), each `[x, y, z, r]` (unit 02's `femoral_heads`). Call `ce_fn(occ, head_xyz, radius, side)` for both cubes. `np.nanmean` of the two sides, unless both are nan.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def ce_error(gt_occ, pred_occ, heads, ce_fn=None):
    """|CE(pred) - CE(gt)| in degrees for each hip.

    heads : (2, 4) rows (left, right): [x, y, z, radius] in mm, cube frame (NaN if the femur was missing)
    ce_fn : the angle function, default xray3d.metrics.ce_angle (the checks pass a stand-in)
    returns {"left": deg, "right": deg, "mean": mean of the finite ones}   (nan where unknown)
    """
    raise NotImplementedError("TODO C1")

In [ ]:
T.check_ce_error(ce_error)

In [ ]:
ROWS = []
for sid in test_ids:
    x = torch.from_numpy(torchdata.standardise(np.stack([ts[sid]["ap"], ts[sid]["lat"]])))[None]
    d, e = [], []
    for m in MODELS[2]:
        occ = predict(m, x)
        d.append(metrics.dice(ts[sid]["occ"], occ))
        e.append(ce_error(ts[sid]["occ"], occ, ts[sid]["femoral_heads"])["mean"])
    ROWS.append({"subject": sid, "dice": float(np.mean(d)), "ce_err": float(np.nanmean(e)) if np.isfinite(e).any() else float("nan")})
    print(f"{sid}: Dice {ROWS[-1]['dice']:.3f}   CE error {ROWS[-1]['ce_err']:5.1f} deg")

### TODO C2 · `error_summary`  ·  suggested driver: **Noa**

Summarise the table: does a better Dice go with a smaller angle error (a **rank** correlation: with 9 patients one
outlier shouldn't decide it), and who are the worst cases by each measure, and by both?

> **Nudge:** Spearman = Pearson on the ranks; tied values share the average rank (`scipy.stats.rankdata` does that). Leave out rows where either number is nan. Worst Dice = lowest first; worst CE = largest first.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def error_summary(rows, k=3):
    """rows: [{"subject", "dice", "ce_err"}, ...]  ->
    {"spearman": rank correlation of dice and ce_err (rows with both finite; nan if < 2),
     "worst_dice": the k subjects with the lowest Dice (worst first),
     "worst_ce": the k subjects with the largest finite ce_err (worst first),
     "both": the subjects in both lists (in worst_dice order)}
    """
    raise NotImplementedError("TODO C2")

In [ ]:
T.check_error_summary(error_summary)

In [ ]:
summ = error_summary(ROWS, k=3)
print(f"Spearman(Dice, CE error) = {summ['spearman']:.2f}    worst Dice: {summ['worst_dice']}    worst CE: {summ['worst_ce']}    both: {summ['both']}")
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter([r["dice"] for r in ROWS], [r["ce_err"] for r in ROWS], color="#4a90c2")
for r in ROWS:
    ax.annotate(r["subject"], (r["dice"], r["ce_err"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set(xlabel="Dice (2 views)", ylabel="CE angle error (deg, mean of both hips)", title="TotalSegmentator test: overlap vs the clinical angle")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
worst = summ["worst_dice"][0]
x = torch.from_numpy(torchdata.standardise(np.stack([ts[worst]["ap"], ts[worst]["lat"]])))[None]
occ = predict(MODELS[2][0], x)
plotting.show_row([ts[worst]["ap"], ts[worst]["lat"], prep.overlap_image(prep.coronal(ts[worst]["occ"]), prep.coronal(occ, int(np.argmax(ts[worst]["occ"].sum(axis=(0, 2))))))],
                  [f"{worst} (worst Dice): AP", "LAT", "truth vs prediction (coronal)"], size=3.6)

**Explain-back (part C):** if Spearman is weak, what does that say about using Dice alone in the report? Look at the
worst case's X-rays: is it unusual (size, position, a cut field, a missing femur), or does the network fail on a normal
pelvis? A CE error of 5° is about one voxel at the acetabular roof (5 mm at ~50 mm from the head centre): what is the
smallest CE error a 64³ cube can resolve?

### Saving the results (aggregate numbers only)

In [ ]:
import csv
# smoke models: the numbers mean nothing, so they go next to the runs (outside git, like unit 06's
# experiment_smoke.csv), never into results/. The real file goes to results/ and is the one you commit.
out = trainer.runs_root() / "sim_vs_real_smoke.csv" if IS_SMOKE else env.results_root() / "sim_vs_real_v1.csv"
out.parent.mkdir(parents=True, exist_ok=True)
tag = {"models": ";".join(USED[1] + USED[2]), "smoke_models_meaningless": IS_SMOKE}
rows = [{"section": "gap", "what": f"{g['from']}->{g['to']} ({g['step']})", "n_views": g["n_views"], "metric": g["metric"],
         "n": g["n"], "mean": round(g["mean"], 4), "sd": round(g["sd"], 4), "worse": g["worse"], **tag} for g in GAPS]
for v in (1, 2):
    for name in deepfluoro.INPUTS:
        for mt in METRICS:
            vals = [scores[v, name][s][mt] for s in SPECIMENS]
            rows.append({"section": "deepfluoro_mean", "what": name, "n_views": v, "metric": mt, "n": len(vals),
                         "mean": round(float(np.mean(vals)), 4), "sd": round(float(np.std(vals, ddof=1)), 4), "worse": "", **tag})
for key, what in (("gt", "reprojection IoU, GT cube vs 2D labels"), ("pred", "reprojection IoU, 2-view real prediction vs 2D labels")):
    vals = [r[key] for r in REPROJ]
    rows.append({"section": "reprojection", "what": what, "n_views": 2, "metric": "iou", "n": len(vals),
                 "mean": round(float(np.mean(vals)), 4), "sd": round(float(np.std(vals, ddof=1)), 4), "worse": "", **tag})
for what, vals in (("TS test r(var, error)", [u["r"] for u in UNC["ts"]]),
                   ("DeepFluoro mean var, drr_pose", [u["drr_pose"]["mean_var"] for u in UNC["df"]]),
                   ("DeepFluoro mean var, real", [u["real"]["mean_var"] for u in UNC["df"]])):
    rows.append({"section": "uncertainty", "what": what, "n_views": 2, "metric": "", "n": len(vals),
                 "mean": round(float(np.nanmean(vals)), 5), "sd": round(float(np.nanstd(vals, ddof=1)), 5), "worse": "", **tag})
for what, val in (("TS test Dice", np.mean([r["dice"] for r in ROWS])), ("TS test CE error (deg)", np.nanmean([r["ce_err"] for r in ROWS])),
                  ("TS test Spearman(Dice, CE error)", summ["spearman"])):
    rows.append({"section": "error_analysis", "what": what, "n_views": 2, "metric": "", "n": len(ROWS),
                 "mean": round(float(val), 4), "sd": "", "worse": "", **tag})
# Licence guard: with ONE specimen an "aggregate" is that cadaver's own numbers, i.e. per-specimen data,
# which CC BY-NC-ND forbids sharing. So the committable file needs at least 2 specimens.
if not IS_SMOKE and len(SPECIMENS) < 2:
    print(f"NOT written: only {len(SPECIMENS)} specimen(s) on disk. An aggregate of one cadaver is that cadaver's "
          "numbers (licence: not shareable). Get at least 2 specimens (deepfluoro.fetch()) and rerun.")
else:
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader(); w.writerows(rows)
    print(f"wrote {len(rows)} aggregate rows to {out}" + ("   (SMOKE MODELS: meaningless, kept outside results/)" if IS_SMOKE else ""))

**Part C done.** Move `ce_error` and `error_summary` into `src/xray3d/realxray.py`. Then run pytest:

In [ ]:
!{sys.executable} -m pytest -q -rs .

Fill in `LOG.md` (driver per TODO, hints, explain-back answers, the numbers). **Before you commit: clear all outputs
and save** (DeepFluoro licence), and commit only `results/sim_vs_real_v1.csv` from this unit's outputs. Next:
**unit 08**, putting it together.